# 00 · Introduction: Strands Agents

Welcome. In this tutorial you will build two working agents with Strands Agents: **ShipFast**, a support agent
that looks up orders and issues refunds (and asks a person before any refund over $100), and a **research
agent** that searches, drafts an answer and critiques its own draft. You will get there one idea at a time,
from a single model call in notebook 01 to the finished agents in notebook 08.

First, though, it helps to know what Strands Agents is and how it thinks about agents, because that explains
most of the design choices you will meet along the way. At the end you will check that your setup works.

## What it is

**Strands Agents** is an open-source SDK from AWS (Apache-2.0 licence), for Python and TypeScript. Its
README describes it as "a simple yet powerful SDK that takes a model-driven approach to building and running
AI agents", from simple conversational assistants to complex autonomous workflows. Among its listed
features: a "simple agent loop that just works and is fully customizable", support for many model
providers (Amazon Bedrock, Anthropic, Gemini, LiteLLM, Llama, Ollama, OpenAI and more), multi-agent systems,
and built-in support for MCP servers.

Being an AWS project shows in one default: with no model given, an agent uses **Amazon Bedrock**, which
needs AWS credentials. Every agent in this tutorial is given the local model explicitly.

> **Versions.** The quotes are from the Strands Agents README (strandsagents.com). Everything in this
> tutorial uses Strands Agents 1.63.0; newer versions may differ in the details.

## The core idea

![Strands Agents: the core idea](images/strands_core_idea.svg)

Strands' idea is **model-driven**: instead of drawing the steps, you give a capable model a system prompt and
tools, and let it plan. The framework keeps the loop small:

- An **Agent** is a model, a system prompt and tools. Calling it runs the loop: the model either answers or
  asks for a tool; the agent runs the tool and hands back the result; repeat until the model answers.
- **Tools** are Python functions with `@tool`; the agent turns their signatures and docstrings into specs.
- **Hooks** run before and after each model or tool call. They are where you log, cap, cancel or pause
  (interrupt) the loop: Strands' answer to guardrails and human approval.
- **Memory** lives in the agent (`agent.messages`); session managers save and restore it.

When fixed steps are needed, Strands connects agents with `GraphBuilder` or lets them hand off in a `Swarm`;
for steps that pass structured data, plain Python around agents is often clearest.

## How the course framed it

The Agentic AI Crash Course does not cover Strands: its framework slides compare LangGraph and CrewAI and
only mention other SDKs in passing. Everything above comes from the Strands project itself, and this
tutorial fills in the rest by building with it.

## What we found building with it

We built the same two agents in LangGraph, CrewAI and Strands, with the same tools, data and local model,
and scored them the same way. Here is how Strands Agents did:

- Results here: ShipFast 4 of 8 and research 9 of 10. The ShipFast gap traced to the tool
  schemas: Strands adds a placeholder description ("Parameter order_id") to every tool parameter, and that
  alone turned the 8B model from calling the refund tool to only describing it. With small models, check
  the tool specs it generates.
- It had the smallest ShipFast agent of the three, and conversation memory built in.
- The approval gate pauses **inside the tool call**, with a hook that interrupts, much like LangGraph.
- There is no default cap on the agent loop (a hook adds one), and asking an agent for structured output
  cannot be forced on Ollama, so the reliable path is the model provider's native structured call.

Treat these as a first look rather than a verdict: each case ran once, with a small model on one laptop.
The full side-by-side is in `COMPARISON.md` at the repo root.

## The concepts, and where you meet them

| Concept | Notebook |
|---|---|
| `Agent`, `OllamaModel`, `structured_output_model` | 01 |
| Tools (`@tool`), tool specs, `agent.messages` | 02 |
| The agent loop, stop reasons, a cap with a hook | 03 |
| The agent's own memory; session managers | 04 |
| `GraphBuilder`, `Swarm`, plain Python around agents | 05 |
| Hooks: `BeforeToolCallEvent`, interrupts, `cancel_tool` | 06 |
| Hooks and result metrics for measurement | 07 |
| Putting it together: ShipFast and the research agent | 08 |

## How this tutorial works

![The path through every tutorial](images/roadmap.svg)

- **Same path in every framework.** The LangGraph, CrewAI and Strands tutorials have the same notebooks in
  the same order, so you can read one notebook in two frameworks side by side. Each notebook ends with
  "How the others do this".
- **Two running examples.** **ShipFast**, a customer-support agent for a made-up shop (orders, refund
  policy, a human approval for refunds over $100, memory across turns), and a **research agent** that answers
  questions about a made-up company from fixed articles, with a critique loop.
- **One shared contract.** The data, tools, instructions, scoring and evaluation runner live once, in
  `contracts/`; this tutorial wraps them in Strands Agents. Its reusable code lives in `strands_agents_impl/`, and both the
  notebooks and `run_eval.py` import it, so the code you read is the code that was tested.
- **One local model.** `qwen3-agents` (Qwen 3 8B) on Ollama, with fixed settings, so differences come from
  the framework. No API key is needed.
- **Real outputs, mistakes included.** Every notebook has been run, and the notes describe what actually
  happened, including where the small model got things wrong. Those mistakes are part of the lesson. When
  you run a notebook yourself, expect the details to differ a little.

## Setup check

You need Ollama running (`ollama serve`) and the shared model created once
(`ollama create qwen3-agents -f contracts/ollama/Modelfile`; see the README). Pick this tutorial's kernel
(`.venv/bin/python` in this tutorial's folder) and run the cell below.

In [1]:
# Check that Ollama is running and the shared model exists. If not, this stops with the command that fixes it.
from importlib.metadata import version

from agent_contracts.harness import load_settings, verify_model

verify_model()

print("Model:         ", load_settings().model)
print("Strands Agents:", version("strands-agents"))

Model:          qwen3-agents
Strands Agents: 1.63.0


If you see the model name and the versions, you are all set. Next up: **01 · The model**, where you make
your first call.